# Paste and render MEI

**CAMAT workflow 1 of 4: handling MEI files.** See `docs/guides/mei-introduction.md` for a beginner introduction and `docs/guides/edition-building.md` for the complete workflow.

Paste or edit a complete MEI document in the text area, then select **Render MEI**. Verovio runs locally through CAMAT, so no score is uploaded to an external editor. The page control can display any page in a multi-page score.


In [1]:
from pathlib import Path
import html
import sys
import xml.etree.ElementTree as ET

import ipywidgets as widgets
from IPython.display import SVG, clear_output, display

# Find the checkout whether Jupyter starts in the repository root or notebooks/.
_here = Path.cwd().resolve()
ROOT = next(
    (path for path in (_here, *_here.parents) if (path / 'camat' / '__init__.py').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from a CAMAT source checkout.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from camat import (
    __version__,
    vrv_load_data,
    vrv_quiet,
    vrv_render_page,
    vrv_set_options,
)

print('CAMAT:', __version__)
print('Repository:', ROOT)


CAMAT: 0.2.0
Repository: /home/egor/Nextcloud/code/public_repos/camat_v2


## MEI editor and score preview

The initial value is a complete one-measure MEI 5.1 document. Replace everything in the editor with your own MEI. A render verifies that the XML is well formed and that Verovio can load it; it is not a substitute for schema or project consistency checks.


In [ ]:
DEFAULT_MEI = '''<mei xmlns="http://www.music-encoding.org/ns/mei" meiversion="5.1">
  <meiHead>
    <fileDesc>
      <titleStmt><title>Minimal MEI example</title></titleStmt>
      <pubStmt/>
    </fileDesc>
  </meiHead>
  <music>
    <body>
      <mdiv>
        <score>
          <scoreDef keysig="1s" meter.count="4" meter.unit="4">
            <staffGrp>
              <staffDef n="1" lines="5" clef.shape="G" clef.line="2"/>
            </staffGrp>
          </scoreDef>
          <section>
            <measure n="1">
              <staff n="1">
                <layer n="1">
                  <note pname="g" oct="4" dur="4" xml:id="m1n1"/>
                  <note pname="a" oct="4" dur="4" accid="s" xml:id="m1n2"/>
                  <note pname="b" oct="4" dur="4" xml:id="m1n3"/>
                  <note pname="c" oct="5" dur="4" xml:id="m1n4"/>
                </layer>
              </staff>
              <dynam startid="#m1n2">f</dynam>
            </measure>
          </section>
        </score>
      </mdiv>
    </body>
  </music>
</mei>'''

mei_editor = widgets.Textarea(
    value=DEFAULT_MEI,
    layout=widgets.Layout(width='100%', height='430px'),
)
render_button = widgets.Button(
    description='Render MEI', button_style='primary', icon='music'
)
page_number = widgets.BoundedIntText(
    value=1, min=1, max=1, description='Page:',
    layout=widgets.Layout(width='150px'),
)
status = widgets.HTML()
score_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #d0d7de', max_height='760px',
        overflow='auto', padding='8px', width='100%',
    )
)
_rendering = False

def render_mei(_=None):
    global _rendering
    if _rendering:
        return
    _rendering = True
    try:
        xml_root = ET.fromstring(mei_editor.value)
        if xml_root.tag.rsplit('}', 1)[-1] != 'mei':
            raise ValueError('The document root must be <mei>. Paste a complete MEI document.')
        with vrv_quiet():
            vrv_set_options(
                pageWidth=9000, pageHeight=12000, scale=35,
                breaks='auto', adjustPageHeight=True,
                footer='none', svgViewBox=True,
            )
            page_count = vrv_load_data(mei_editor.value, input_from='mei')
            if page_count < 1:
                raise RuntimeError('Verovio loaded no renderable score pages.')
            page_number.max = page_count
            if page_number.value > page_count:
                page_number.value = page_count
            svg = vrv_render_page(page_number.value)
        with score_output:
            clear_output(wait=True)
            display(SVG(svg))
        status.value = (
            f'<span style="color:#1a7f37">Rendered page {page_number.value} '
            f'of {page_count}.</span>'
        )
    except Exception as exc:
        with score_output:
            clear_output(wait=True)
        status.value = (
            '<span style="color:#cf222e"><strong>Could not render MEI:</strong> '
            f'{html.escape(str(exc))}</span>'
        )
    finally:
        _rendering = False

render_button.on_click(render_mei)
page_number.observe(render_mei, names='value')
display(widgets.VBox([
    mei_editor, widgets.HBox([render_button, page_number]),
    status, score_output,
]))
render_mei()


## Optional: save the current MEI text

Set `SAVE_PATH` to a file path when you want to keep the edited text. The default is `None`, so executing the notebook does not write a file.


In [ ]:
SAVE_PATH = None  # Example: ROOT / 'my_score.mei'

if SAVE_PATH is not None:
    save_path = Path(SAVE_PATH).expanduser().resolve()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    save_path.write_text(mei_editor.value, encoding='utf-8')
    print('Saved:', save_path)
else:
    print('Nothing saved; set SAVE_PATH when you want to write the current MEI.')


## What was produced?

- the MEI remains editable as text in the notebook;
- Verovio loads the current text and renders the selected page as SVG;
- no network service receives the score;
- no file is written unless `SAVE_PATH` is set explicitly.

Next, use the Workflow 1 consistency checks to validate the MEI, or open `mei_facsimile_viewer.ipynb` when the file contains facsimile surfaces and measure-zone links.
